# Phase 2 Assessment — Portfolio Risk Emulator

Run on **SageMaker Studio** `ml.g4dn.xlarge` with the PyTorch 2.x DLC image.

Covers every item on the Phase 2 checklist in `progress.md`:
- Data quality
- Training convergence
- NN accuracy vs MC ground truth
- Tail calibration
- Inference speedup
- Regime stress tests

## 0. Setup

In [ ]:
# SageMaker DLC already has numpy, pandas, matplotlib, seaborn, pyarrow, scikit-learn.
# Only cupy is missing — install without touching existing packages to avoid numpy conflicts.
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "cupy-cuda12x", "fastrlock", "--no-deps"])

In [ ]:
import os, sys, glob, io, pickle, time, math, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from torch.utils.data import DataLoader, Dataset, random_split
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="darkgrid", palette="muted")

# Add repo root to path so training module imports work
sys.path.insert(0, os.path.abspath(".."))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# --- Configure these before running ---
S3_DATA_URI   = os.environ.get("S3_DATA_URI",  "s3://YOUR_BUCKET/data/")   # where shards land
S3_MODEL_URI  = os.environ.get("S3_MODEL_URI", "s3://YOUR_BUCKET/models/v1/")  # where model lands
LOCAL_DATA    = "/tmp/risk_data/"    # local scratch for parquet shards
LOCAL_MODELS  = "/tmp/risk_models/"  # local scratch for model artifacts
N_SCENARIOS   = 50_000   # reduce to 5_000 for a quick smoke test
EPOCHS        = 50

os.makedirs(LOCAL_DATA, exist_ok=True)
os.makedirs(LOCAL_MODELS, exist_ok=True)

## 1. Generate Scenarios

In [ ]:
# Run scenario generation — writes Parquet shards to LOCAL_DATA (and optionally S3)
# Use --out-prefix '' to write locally; set to S3_DATA_URI to also push to S3
result = subprocess.run(
    [sys.executable, "generate_scenarios.py",
     "--n-scenarios", str(N_SCENARIOS),
     "--shard-size", "10000",
     "--out-prefix", LOCAL_DATA],
    capture_output=True, text=True, cwd=os.path.dirname(os.path.abspath("."))
)
print(result.stdout[-3000:] if result.stdout else "")
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])

## 2. Data Quality

In [ ]:
from training.train import FEATURE_COLS, TARGET_COLS

shards = sorted(glob.glob(os.path.join(LOCAL_DATA, "*.parquet")))
assert shards, f"No parquet shards found in {LOCAL_DATA}"
df = pd.concat([pd.read_parquet(p) for p in shards], ignore_index=True)
print(f"Loaded {len(df):,} rows from {len(shards)} shards")
df.head(3)

In [ ]:
# --- Check: no NaN or Inf in features or targets ---
feat_df  = df[FEATURE_COLS]
tgt_df   = df[TARGET_COLS]

nan_feats = feat_df.isna().sum().sum()
inf_feats = np.isinf(feat_df.values).sum()
nan_tgts  = tgt_df.isna().sum().sum()
inf_tgts  = np.isinf(tgt_df.values).sum()

status = lambda ok: "PASS" if ok else "FAIL"
print(f"Feature NaN count : {nan_feats}  [{status(nan_feats == 0)}]")
print(f"Feature Inf count : {inf_feats}  [{status(inf_feats == 0)}]")
print(f"Target  NaN count : {nan_tgts}   [{status(nan_tgts == 0)}]")
print(f"Target  Inf count : {inf_tgts}   [{status(inf_tgts == 0)}]")

In [ ]:
# --- Check: target ordering constraints ---
# VaR < 0 for most rows (loss is negative P&L)
# ES <= VaR  (ES is a worse loss)
# q05 < mean_pnl < q95

var_neg_rate   = (df["var"] < 0).mean()
es_leq_var     = (df["es"] <= df["var"]).mean()
q05_lt_mean    = (df["q05_pnl"] < df["mean_pnl"]).mean()
mean_lt_q95    = (df["mean_pnl"] < df["q95_pnl"]).mean()

print(f"VaR < 0 rate      : {var_neg_rate:.3f}  [expect > 0.90]  [{status(var_neg_rate > 0.90)}]")
print(f"ES <= VaR rate    : {es_leq_var:.3f}  [expect = 1.00]  [{status(es_leq_var > 0.99)}]")
print(f"q05 < mean rate   : {q05_lt_mean:.3f}  [expect = 1.00]  [{status(q05_lt_mean > 0.99)}]")
print(f"mean < q95 rate   : {mean_lt_q95:.3f}  [expect = 1.00]  [{status(mean_lt_q95 > 0.99)}]")

In [ ]:
# --- Check: scenario diversity (horizon & confidence coverage) ---
print("Horizon distribution:")
print(df["log_horizon"].apply(lambda x: round(math.exp(x))).value_counts().sort_index())
print("\nConfidence distribution:")
print(df["confidence"].value_counts().sort_index())

In [ ]:
# --- Plot: feature and target distributions ---
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
fig.suptitle("Key feature & target distributions", fontsize=13)

plot_feats = ["realized_vol_21d", "atm_implied_vol", "risk_free_rate", "confidence"]
for ax, col in zip(axes[0], plot_feats):
    ax.hist(df[col].dropna(), bins=40, edgecolor="none")
    ax.set_title(col, fontsize=9)

for ax, col in zip(axes[1], TARGET_COLS[:4]):
    ax.hist(df[col].dropna(), bins=40, edgecolor="none", color="coral")
    ax.set_title(col, fontsize=9)

plt.tight_layout()
plt.show()

## 3. Training (instrumented)

In [ ]:
# Inline the training loop with per-epoch qc_loss tracking
# (train.py only logs every 10 epochs and doesn't separate qc_loss)

from training.train import RiskDataset, RiskEmulator, quantile_consistency_loss

X_raw = df[FEATURE_COLS].values.astype(np.float32)
y_raw = df[TARGET_COLS].values.astype(np.float32)

scaler = StandardScaler()
X = scaler.fit_transform(X_raw).astype(np.float32)

dataset  = RiskDataset(X, y_raw)
val_size = max(1, int(0.1 * len(dataset)))
train_ds, val_ds = random_split(dataset, [len(dataset) - val_size, val_size])

train_loader = DataLoader(train_ds, batch_size=2048, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=8192)

model     = RiskEmulator().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
mse_fn    = nn.MSELoss()

history = {"train": [], "val": [], "qc": []}
best_val, best_state = float("inf"), {}

for epoch in range(1, EPOCHS + 1):
    model.train()
    tloss, qcloss = 0.0, 0.0
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
        optimizer.zero_grad()
        pred = model(X_b)
        qc   = quantile_consistency_loss(pred)
        loss = mse_fn(pred, y_b) + 0.1 * qc
        loss.backward()
        optimizer.step()
        tloss  += mse_fn(pred, y_b).item() * len(X_b)
        qcloss += qc.item()             * len(X_b)
    tloss  /= len(train_ds)
    qcloss /= len(train_ds)

    model.eval()
    vloss = 0.0
    with torch.no_grad():
        for X_b, y_b in val_loader:
            X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
            vloss += mse_fn(model(X_b), y_b).item() * len(X_b)
    vloss /= len(val_ds)
    scheduler.step()

    history["train"].append(tloss)
    history["val"].append(vloss)
    history["qc"].append(qcloss)

    if vloss < best_val:
        best_val   = vloss
        best_state = {k: v.clone() for k, v in model.state_dict().items()}

    if epoch % 10 == 0:
        print(f"epoch {epoch:3d}/{EPOCHS}  train={tloss:.4f}  val={vloss:.4f}  qc={qcloss:.4f}")

model.load_state_dict(best_state)
model.eval()
print(f"\nBest val loss: {best_val:.4f}")

## 4. Convergence Analysis

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

epochs_x = range(1, EPOCHS + 1)
ax1.plot(epochs_x, history["train"], label="train")
ax1.plot(epochs_x, history["val"],   label="val")
ax1.set_title("MSE Loss")
ax1.set_xlabel("epoch")
ax1.legend()

ax2.plot(epochs_x, history["qc"], color="darkorange")
ax2.set_title("Quantile Consistency Penalty")
ax2.set_xlabel("epoch")
ax2.set_ylabel("qc_loss")

plt.tight_layout()
plt.show()

# --- Convergence checks ---
final_qc        = history["qc"][-1]
train_val_ratio = history["train"][-1] / (history["val"][-1] + 1e-9)
val_monotone    = all(history["val"][i] >= history["val"][i+1]
                      for i in range(len(history["val"]) - 2))  # allow 1 uptick

print(f"Val loss monotonically decreasing : {val_monotone}         [{status(val_monotone)}]")
print(f"Train/val ratio (final)           : {train_val_ratio:.2f}  [expect < 2.0]  [{status(train_val_ratio < 2.0)}]")
print(f"QC penalty at final epoch         : {final_qc:.5f}  [expect near 0] [{status(final_qc < 0.001)}]")

## 5. NN Accuracy vs MC Ground Truth

In [ ]:
# Holdout evaluation — use val_ds (10% of data, unseen during training)
val_indices = val_ds.indices
X_val = torch.tensor(X[val_indices]).to(DEVICE)
y_val = y_raw[val_indices]

with torch.no_grad():
    pred_val = model(X_val).cpu().numpy()

def r2(pred, true):
    return float(1 - np.var(pred - true) / (np.var(true) + 1e-8))

r2_var = r2(pred_val[:, 0], y_val[:, 0])
r2_es  = r2(pred_val[:, 1], y_val[:, 1])

mae_by_target = {col: float(np.abs(pred_val[:, i] - y_val[:, i]).mean())
                 for i, col in enumerate(TARGET_COLS)}

mean_notional  = 1.0  # normalized; adjust if notional is in features
mae_var_pct    = mae_by_target["var"] / (abs(y_val[:, 0]).mean() + 1e-8)
over_est_rate  = float((pred_val[:, 0] > y_val[:, 0]).mean())

print(f"VaR  R²                  : {r2_var:.4f}   [threshold ≥ 0.90]  [{status(r2_var >= 0.90)}]")
print(f"ES   R²                  : {r2_es:.4f}   [threshold ≥ 0.85]  [{status(r2_es >= 0.85)}]")
print(f"MAE(VaR) / mean|VaR|     : {mae_var_pct:.4f}  [threshold < 0.02]  [{status(mae_var_pct < 0.02)}]")
print(f"VaR over-estimation rate : {over_est_rate:.3f}   [threshold < 0.10]  [{status(over_est_rate < 0.10)}]")
print("\nMAE by target:")
for col, err in mae_by_target.items():
    print(f"  {col:<14} {err:.5f}")

In [ ]:
# --- Scatter plots: predicted vs actual for VaR and ES ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

sample = np.random.choice(len(y_val), min(3000, len(y_val)), replace=False)

for ax, col_idx, title in [(ax1, 0, "VaR"), (ax2, 1, "ES")]:
    true_s = y_val[sample, col_idx]
    pred_s = pred_val[sample, col_idx]
    ax.scatter(true_s, pred_s, alpha=0.25, s=8)
    lims = [min(true_s.min(), pred_s.min()), max(true_s.max(), pred_s.max())]
    ax.plot(lims, lims, "r--", lw=1, label="perfect")
    ax.set_xlabel(f"MC {title}")
    ax.set_ylabel(f"NN {title}")
    ax.set_title(f"{title}  R²={r2(pred_val[:, col_idx], y_val[:, col_idx]):.3f}")
    ax.legend()

plt.tight_layout()
plt.show()

## 6. Tail Calibration

In [ ]:
# Calibration: at 95%/99% confidence, the fraction of rows where pred_var > mc_var
# should be approximately (1 - confidence), i.e., 0.05 and 0.01

df_val = df.iloc[val_indices].reset_index(drop=True)

for conf, tol in [(0.95, 0.02), (0.99, 0.03)]:
    mask  = np.abs(df_val["confidence"].values - conf) < 0.001
    if mask.sum() == 0:
        print(f"No {conf} confidence rows in val set — skipping")
        continue
    over  = (pred_val[mask, 0] > y_val[mask, 0]).mean()
    target = 1 - conf
    ok    = abs(over - target) <= tol
    print(f"Conf={conf}: over-est rate={over:.3f}  target={target:.2f} ±{tol}  [{status(ok)}]")

In [ ]:
# ES error by percentile — should be concentrated in body, not tails
es_err = pred_val[:, 1] - y_val[:, 1]
pctiles = np.percentile(y_val[:, 1], [0, 10, 25, 50, 75, 90, 100])

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(es_err, bins=60, edgecolor="none")
ax.axvline(0, color="red", lw=1.5, linestyle="--")
ax.set_title("ES prediction error (pred - true)")
ax.set_xlabel("error")
ax.set_ylabel("count")
plt.tight_layout()
plt.show()

print(f"ES error p5  : {np.percentile(es_err, 5):.4f}")
print(f"ES error p50 : {np.percentile(es_err, 50):.4f}")
print(f"ES error p95 : {np.percentile(es_err, 95):.4f}")

## 7. Inference Speedup

In [ ]:
BATCH = 10_000
X_bench = torch.tensor(X[:BATCH], dtype=torch.float32)

# GPU batch
if DEVICE.type == "cuda":
    X_gpu = X_bench.to(DEVICE)
    with torch.no_grad():  # warmup
        _ = model(X_gpu)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        _ = model(X_gpu)
    torch.cuda.synchronize()
    gpu_ms = (time.perf_counter() - t0) * 1000
else:
    gpu_ms = None

# CPU batch
model_cpu = model.cpu()
with torch.no_grad():  # warmup
    _ = model_cpu(X_bench)
t0 = time.perf_counter()
with torch.no_grad():
    _ = model_cpu(X_bench)
cpu_ms = (time.perf_counter() - t0) * 1000
model   = model.to(DEVICE)

# Single-scenario latency (CPU)
X_single = X_bench[:1]
latencies = []
for _ in range(100):
    t0 = time.perf_counter()
    with torch.no_grad():
        _ = model_cpu(X_single)
    latencies.append((time.perf_counter() - t0) * 1000)
single_ms = np.median(latencies)

print(f"CPU batch {BATCH:,} scenarios  : {cpu_ms:.1f}ms   [threshold < 50ms]  [{status(cpu_ms < 50)}]")
if gpu_ms is not None:
    print(f"GPU batch {BATCH:,} scenarios  : {gpu_ms:.1f}ms   [threshold < 10ms]  [{status(gpu_ms < 10)}]")
print(f"Single-scenario CPU latency   : {single_ms:.2f}ms  [threshold < 5ms]   [{status(single_ms < 5)}]")

## 8. Regime Stress Tests

In [ ]:
def eval_subset(mask_name: str, mask: np.ndarray) -> None:
    n = mask.sum()
    if n < 10:
        print(f"{mask_name}: only {n} samples — skipping")
        return
    X_sub = torch.tensor(X[mask], dtype=torch.float32).to(DEVICE)
    y_sub = y_raw[mask]
    with torch.no_grad():
        pred_sub = model(X_sub).cpu().numpy()
    mae_var = float(np.abs(pred_sub[:, 0] - y_sub[:, 0]).mean())
    mean_abs_var = abs(y_sub[:, 0]).mean()
    mae_pct  = mae_var / (mean_abs_var + 1e-8)
    r2_sub   = r2(pred_sub[:, 0], y_sub[:, 0])
    print(f"{mask_name:<35} n={n:>6,}  R²={r2_sub:.3f}  MAE(VaR)/mean|VaR|={mae_pct:.3f}")

# Reconstruct original (unscaled) feature values for masking
realized_vol = df["realized_vol_21d"].values
log_horizon  = df["log_horizon"].values
option_wt    = df["option_weight"].values

eval_subset("Full dataset",                       np.ones(len(df), dtype=bool))
eval_subset("High-vol (realized_vol_21d > 0.12)", realized_vol > 0.12)
eval_subset("Long-horizon (63-day)",              np.abs(log_horizon - math.log(63)) < 0.01)
eval_subset("Options-heavy (option_weight > 0.5)",option_wt > 0.5)

In [ ]:
# High-vol vs low-vol error comparison
hi_vol_mask = realized_vol > 0.12
lo_vol_mask = ~hi_vol_mask

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, mask, label in [(axes[0], lo_vol_mask, "Low-vol"), (axes[1], hi_vol_mask, "High-vol")]:
    if mask.sum() == 0:
        continue
    X_sub  = torch.tensor(X[mask], dtype=torch.float32).to(DEVICE)
    y_sub  = y_raw[mask]
    with torch.no_grad():
        p_sub  = model(X_sub).cpu().numpy()
    err = p_sub[:, 0] - y_sub[:, 0]
    ax.hist(err, bins=50, edgecolor="none")
    ax.axvline(0, color="red", lw=1.2, linestyle="--")
    ax.set_title(f"{label} VaR error (n={mask.sum():,})")
    ax.set_xlabel("pred - true VaR")

plt.tight_layout()
plt.show()

## 9. Export & Push to S3

In [ ]:
import boto3

scripted = torch.jit.script(model.cpu())
torch.jit.save(scripted, os.path.join(LOCAL_MODELS, "model.pt"))
with open(os.path.join(LOCAL_MODELS, "scaler.pkl"), "wb") as f:
    pickle.dump(scaler, f)

print(f"Artifacts saved locally to {LOCAL_MODELS}")

if S3_MODEL_URI and not S3_MODEL_URI.startswith("s3://YOUR_"):
    bucket, prefix = S3_MODEL_URI.replace("s3://", "").split("/", 1)
    s3 = boto3.client("s3")
    for fname in ["model.pt", "scaler.pkl"]:
        local_path = os.path.join(LOCAL_MODELS, fname)
        s3.upload_file(local_path, bucket, f"{prefix}{fname}")
        print(f"Uploaded s3://{bucket}/{prefix}{fname}")
else:
    print("S3_MODEL_URI not configured — skipping S3 upload")

## 10. Pass / Fail Summary

In [ ]:
rows = [
    ("Data quality",     "No NaN/Inf in features",             nan_feats == 0 and inf_feats == 0),
    ("Data quality",     "No NaN/Inf in targets",              nan_tgts == 0 and inf_tgts == 0),
    ("Data quality",     "ES <= VaR rate > 0.99",              es_leq_var > 0.99),
    ("Data quality",     "q05 < mean < q95 rate > 0.99",       q05_lt_mean > 0.99 and mean_lt_q95 > 0.99),
    ("Convergence",      "Val loss monotonically decreasing",  val_monotone),
    ("Convergence",      "Train/val ratio < 2.0",              train_val_ratio < 2.0),
    ("Convergence",      "QC penalty near 0 at end",           final_qc < 0.001),
    ("NN accuracy",      "VaR R² >= 0.90",                     r2_var >= 0.90),
    ("NN accuracy",      "ES  R² >= 0.85",                     r2_es  >= 0.85),
    ("NN accuracy",      "MAE(VaR)/mean|VaR| < 0.02",          mae_var_pct < 0.02),
    ("NN accuracy",      "VaR over-estimation rate < 0.10",    over_est_rate < 0.10),
    ("Speed",            "CPU batch 10K < 50ms",               cpu_ms < 50),
    ("Speed",            "Single-scenario < 5ms",              single_ms < 5),
]
if gpu_ms is not None:
    rows.append(("Speed", "GPU batch 10K < 10ms", gpu_ms < 10))

summary = pd.DataFrame(rows, columns=["Category", "Check", "Pass"])
summary["Result"] = summary["Pass"].map({True: "PASS", False: "FAIL"})

def highlight(row):
    color = "background-color: #c8f7c5" if row["Pass"] else "background-color: #f7c5c5"
    return ["", "", "", color]

display(summary.style.apply(highlight, axis=1))

n_pass = summary["Pass"].sum()
n_total = len(summary)
print(f"\n{n_pass}/{n_total} checks passed")
if n_pass == n_total:
    print("All checks passed — Phase 2 complete, ready for Phase 3.")
else:
    failing = summary[~summary["Pass"]]["Check"].tolist()
    print("Failing checks:")
    for c in failing:
        print(f"  - {c}")